In [0]:

from pyspark.sql import functions as F

synthetic_order_ids = (
    spark.read.option("header", "true")
    .csv("/Volumes/workspace/olist_landing/batched_files/batch_streaming/orders/*.csv")
    .select("order_id")
    .distinct()
)

deliveries_preview = (
    spark.table("workspace.olist_silver.orders_customer")
    .join(synthetic_order_ids, "order_id", "left_semi")
    .filter(
        (F.col("order_status") == "approved")
        & (F.col("order_estimated_delivery_date") <= F.current_timestamp())
    )
    .select(
        "order_id",
        "order_status",
        "order_delivered_customer_date",
        F.col("order_estimated_delivery_date").alias("simulated_delivery_at"),
    )
)

print("Pedidos distintos elegíveis:", deliveries_preview.select("order_id").distinct().count())
display(deliveries_preview.orderBy("simulated_delivery_at", "order_id").limit(20))

In [0]:
from pyspark.sql import functions as F

deliveries_preview = (
    spark.table("workspace.olist_silver.orders_customer")
    .join(synthetic_order_ids, "order_id", "left_semi")
    .filter(F.col("order_status").isin("approved", "shipped"))
    .withColumn(
        "planned_delivery_at",
        F.when(
            F.pmod(F.xxhash64("order_id", F.lit("delivery_delay")), F.lit(5)) == 0,
            F.expr("order_estimated_delivery_date + INTERVAL 2 DAYS"),
        ).otherwise(F.col("order_estimated_delivery_date")),
    )
    .withColumn(
        "planned_shipment_at",
        F.coalesce(
            F.col("order_delivered_carrier_date"),
            F.greatest(
                F.col("order_approved_at"),
                F.expr("order_estimated_delivery_date - INTERVAL 2 DAYS"),
            ),
        ),
    )
    .filter(
        F.col("order_approved_at").isNotNull()
        & F.col("planned_shipment_at").between(
            F.col("order_approved_at"), F.col("planned_delivery_at")
        )
    )
    .withColumn(
        "simulated_status",
        F.when(F.col("planned_delivery_at") <= F.current_timestamp(), "delivered")
        .when(F.col("planned_shipment_at") <= F.current_timestamp(), "shipped")
        .otherwise(F.col("order_status")),
    )
    .select(
        "order_id",
        F.col("order_status").alias("current_status"),
        "simulated_status",
        "order_approved_at",
        "order_estimated_delivery_date",
        "planned_delivery_at",
        F.when(
            F.col("planned_shipment_at") <= F.current_timestamp(),
            F.col("planned_shipment_at"),
        ).alias("simulated_shipped_at"),
        F.when(
            F.col("simulated_status") == "delivered",
            F.col("planned_delivery_at"),
        ).alias("simulated_delivery_at"),
    )
)

display(
    deliveries_preview.groupBy("current_status", "simulated_status")
    .agg(F.countDistinct("order_id").alias("pedidos"))
)

display(
    deliveries_preview
    .filter(F.col("planned_delivery_at") > F.col("order_estimated_delivery_date"))
    .orderBy("order_estimated_delivery_date", "order_id")
    .limit(20)
)

In [0]:
from delta.tables import DeltaTable
from pyspark.sql import functions as F

orders_table = "workspace.olist_silver.orders_customer"

delivery_updates = (
    deliveries_preview
    .filter(F.col("current_status") != F.col("simulated_status"))
    .select(
        "order_id",
        "current_status",
        "simulated_status",
        "simulated_shipped_at",
        "simulated_delivery_at",
    )
    .distinct()
)

duplicate_ids = delivery_updates.groupBy("order_id").count().filter("count > 1")
if duplicate_ids.limit(1).count():
    raise ValueError("Há pedidos com mais de uma atualização simulada. Confira os IDs.")

merge_result = (
    DeltaTable.forName(spark, orders_table)
    .alias("target")
    .merge(
        delivery_updates.alias("updates"),
        "target.order_id = updates.order_id",
    )
    .whenMatchedUpdate(
        condition="target.order_status = updates.current_status",
        set={
            "order_status": "updates.simulated_status",
            "order_delivered_carrier_date": "updates.simulated_shipped_at",
            "order_delivered_customer_date": "updates.simulated_delivery_at",
            "delivery_date_missing_flag": "false",
            "approval_after_delivery_flag": "false",
            "diff_hours": "CAST(0 AS BIGINT)",
        },
    )
    .execute()
)

display(merge_result)